# Giorno 1 — Object Detection & Multi-Object Tracking

**Obiettivi della giornata:**
1. Rilevare oggetti in video usando YOLOv8 (modello pre-addestrato)
2. Tracciare più oggetti simultaneamente con BoxMOT
3. Simulare condizioni di acquisizione reali tramite degradazione controllata (OpenCV)
4. Confrontare i risultati dei punti 1-2 prima e dopo la degradazione

---
**Dataset:**
- `MOT20` — sequenze di persone in ambienti affollati
- `UA-DETRAC` (`MVI_*`) — sequenze di traffico veicolare

**Modelli:** YOLOv8n (nano, ~6MB) + BoxMOT (ByteTrack)

> ℹ️ In questo laboratorio utilizziamo esclusivamente modelli **pre-addestrati**: nessun training, solo inference.

## 0. Setup — Installazione e import

Eseguire questa cella **una sola volta** all'inizio della sessione.

In [ ]:
# Installazione delle dipendenze (e clone del repo su Google Colab)
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Corso-Computer-Vision'):
        !git clone https://github.com/SalvoSamba01/Corso-Computer-Vision
    %cd /content/Corso-Computer-Vision

!pip install -q ultralytics boxmot opencv-python-headless matplotlib tqdm

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import subprocess
from pathlib import Path
from tqdm import tqdm
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

import sys, os
if os.path.exists('/content/Corso-Computer-Vision'):
    sys.path.append('/content/Corso-Computer-Vision/utils')
else:
    sys.path.append('../utils')
from cv_utils import (
    mostra_frame, mostra_confronto, mostra_griglia,
    stampa_info_video, estrai_frame,
    degrada_immagine, riduci_risoluzione, applica_blur, aggiungi_rumore,
    mostra_confronto
)


def converti_video_h264(input_path: str, output_path: str) -> str:
    """Re-encode video to H.264 for browser playback in Colab."""
    r = subprocess.run(
        ['ffmpeg', '-y', '-i', input_path, '-vcodec', 'libx264',
         '-crf', '23', '-movflags', '+faststart', output_path, '-loglevel', 'quiet'],
        capture_output=True
    )
    if r.returncode == 0 and os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        if input_path != output_path:
            try:
                os.remove(input_path)
            except Exception:
                pass
        return output_path
    return input_path


def mostra_video_colab(path, width=700):
    """Embed a video inline in a Colab/Jupyter cell."""
    from base64 import b64encode
    with open(path, 'rb') as f:
        video_b64 = b64encode(f.read()).decode()
    display(HTML(
        f'<video width="{width}" controls>'
        f'<source src="data:video/mp4;base64,{video_b64}" type="video/mp4">'
        f'</video>'
    ))


print('✅ Import completati')

In [ ]:
# ── Configurazione percorsi ──────────────────────────────────────────────────
import os
from pathlib import Path

if os.path.exists('/content/Corso-Computer-Vision'):
    DATA_DIR = Path('/content/Corso-Computer-Vision/data/day_1')
    BASE_DIR  = Path('/content/Corso-Computer-Vision')
else:
    DATA_DIR = Path('../data/day_1')
    BASE_DIR  = Path('..')

MOT20_DIR  = DATA_DIR / 'MOT20'
DETRAC_DIR = DATA_DIR / 'UA-DETRAC'

VIDEOS_DIR = BASE_DIR / 'results' / 'videos'
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# Verifica dei file disponibili
video_mot20  = sorted(MOT20_DIR.glob('*.mp4'))
video_detrac = sorted(DETRAC_DIR.glob('*.mp4'))
video_files  = video_mot20 + video_detrac

print(f'Video MOT20 trovati: {len(video_mot20)}')
for v in video_mot20:
    print(f'  {v.name}')
print(f'\nVideo UA-DETRAC trovati: {len(video_detrac)}')
for v in video_detrac:
    print(f'  {v.name}')
print(f'\nOutput video → {VIDEOS_DIR}')

---
## 1. Esplorazione del dataset

Prima di applicare qualsiasi modello, esploriamo i video: risoluzione, FPS, contenuto.

**I dataset che utilizziamo:**
- **MOT20** (`MOT20-*.mp4`): benchmark per il tracking di pedoni in scene affollate (stazioni, eventi pubblici)
- **UA-DETRAC** (`MVI_*.mp4`): traffico veicolare ripreso da telecamere di sorveglianza stradali

In [ ]:
# Stampa le informazioni di tutti i video
print('=' * 55)
for v in video_files:
    print(f'\n📹 {v.name}')
    stampa_info_video(str(v))
print('=' * 55)

In [ ]:
# Visualizziamo il primo frame di ciascuna sequenza
frames_preview = []
titoli_preview = []

for v in video_files:
    frame = estrai_frame(str(v), n=1)[0]
    frames_preview.append(frame)
    titoli_preview.append(v.stem)

mostra_griglia(frames_preview, titoli_preview, colonne=4)

In [ ]:
# Selezioniamo un video per tipo come riferimento per gli esperimenti
VIDEO_PEDONI  = str(MOT20_DIR  / 'MOT20-01.mp4')   # benchmark pedoni (affollamento moderato)
VIDEO_VEICOLI = str(DETRAC_DIR / 'MVI_20035.mp4')  # traffico veicolare
VIDEO_FOLLA   = str(MOT20_DIR  / 'MOT20-06.mp4')   # folla densa — sequenza alternativa MOT20

print('Video selezionati per gli esperimenti:')
print(f'  Pedoni  → {Path(VIDEO_PEDONI).name}')
print(f'  Veicoli → {Path(VIDEO_VEICOLI).name}')
print(f'  Folla   → {Path(VIDEO_FOLLA).name}')

---
## 2. Object Detection con YOLOv8

**YOLO** (You Only Look Once) è una famiglia di modelli di object detection in tempo reale.
La versione 8 di Ultralytics è lo stato dell'arte per applicazioni pratiche: veloce, accurata, facile da usare.

**Come funziona (in breve):**
- Il modello divide l'immagine in una griglia
- Per ogni cella predice bounding box + classe + confidenza
- Una fase di NMS (Non-Maximum Suppression) elimina le detection ridondanti

Usiamo `YOLOv8n` (nano): il più leggero della famiglia, ideale per dimostrazione.
I pesi (~6MB) vengono scaricati automaticamente al primo utilizzo.

In [ ]:
from ultralytics import YOLO

# Caricamento del modello (download automatico al primo utilizzo)
# yolov8n = nano (~6MB), yolov8s = small (~22MB), yolov8m = medium (~52MB)
model_yolo = YOLO('yolov8n.pt')
print(f'Modello caricato: {model_yolo.model.__class__.__name__}')
print(f'Classi COCO: {len(model_yolo.names)} classi')

In [ ]:
# Stampa tutte le classi COCO riconosciute dal modello
# Le classi evidenziate (★) sono quelle rilevanti per questo corso
CLASSI_INTERESSE = {0, 1, 2, 3, 5, 7}  # person, bicycle, car, motorcycle, bus, truck

print(f'Classi COCO: {len(model_yolo.names)} classi totali\n')
print(f"{'ID':>3}  {'Nome':<22}  Note")
print('─' * 40)
for idx in sorted(model_yolo.names.keys()):
    nome = model_yolo.names[idx]
    tag = '  ★ (usata nel corso)' if idx in CLASSI_INTERESSE else ''
    print(f'{idx:>3}  {nome:<22}{tag}')

In [ ]:
def disegna_detection(frame: np.ndarray, risultati, 
                      soglia_conf: float = 0.3,
                      classi_filtro: list = None) -> np.ndarray:
    """
    Disegna i bounding box delle detection su un frame.
    
    Args:
        frame: immagine BGR
        risultati: output di YOLO
        soglia_conf: confidenza minima per mostrare la detection
        classi_filtro: lista di ID classe da visualizzare (None = tutte)
    """
    out = frame.copy()
    colori = {0: (0,255,0), 1:(255,165,0), 2:(0,0,255),
               3:(255,0,255), 5:(0,165,255), 7:(0,80,255)}
    
    for box in risultati[0].boxes:
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        if conf < soglia_conf:
            continue
        if classi_filtro is not None and cls_id not in classi_filtro:
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        colore = colori.get(cls_id, (200, 200, 200))
        nome_classe = model_yolo.names[cls_id]
        cv2.rectangle(out, (x1, y1), (x2, y2), colore, 2)
        etichetta = f'{nome_classe} {conf:.2f}'
        cv2.putText(out, etichetta, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, colore, 2)
    return out

In [ ]:
# ── 2.1 Detection su frame singolo: pedoni ───────────────────────────────────
frame_pedoni = estrai_frame(VIDEO_PEDONI, n=1)[0]

risultati = model_yolo(frame_pedoni, verbose=False)
frame_annotato = disegna_detection(frame_pedoni, risultati, 
                                    soglia_conf=0.3, classi_filtro=[0])  # classe 0 = person

n_persone = sum(1 for b in risultati[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) > 0.3)
print(f'Persone rilevate: {n_persone}')
mostra_confronto(frame_pedoni, frame_annotato, 
                 'Originale', f'Detection ({n_persone} persone)')

In [ ]:
# ── 2.2 Detection su frame singolo: veicoli ──────────────────────────────────
frame_veicoli = estrai_frame(VIDEO_VEICOLI, n=1)[0]

risultati_v = model_yolo(frame_veicoli, verbose=False)
frame_annotato_v = disegna_detection(frame_veicoli, risultati_v,
                                      soglia_conf=0.3, classi_filtro=[2, 5, 7])  # car, bus, truck

n_veicoli = sum(1 for b in risultati_v[0].boxes
                if int(b.cls[0]) in [2,5,7] and float(b.conf[0]) > 0.3)
print(f'Veicoli rilevati: {n_veicoli}')
mostra_confronto(frame_veicoli, frame_annotato_v,
                 'Originale', f'Detection ({n_veicoli} veicoli)')

In [ ]:
# ── 2.3 Effetto della soglia di confidenza ───────────────────────────────────
# Mostriamo come la soglia conf cambia il numero di detection

frame_ref = estrai_frame(VIDEO_PEDONI, n=1)[0]
soglie = [0.1, 0.3, 0.5, 0.7]
frame_confronto = []
titoli_confronto = []

ris = model_yolo(frame_ref, verbose=False)
for s in soglie:
    ann = disegna_detection(frame_ref, ris, soglia_conf=s, classi_filtro=[0])
    n = sum(1 for b in ris[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) > s)
    frame_confronto.append(ann)
    titoli_confronto.append(f'conf ≥ {s}  ({n} det.)')

mostra_griglia(frame_confronto, titoli_confronto, colonne=2)

### 🔍 Domanda
Cosa osservate al variare della soglia di confidenza? Qual è il trade-off tra **falsi positivi** e **falsi negativi**?

*(Discutere con il gruppo prima di procedere)*

In [ ]:
# ── 2.4 Detection su una sequenza di frame (mini-video) ──────────────────────
# Processiamo i primi N frame del video e salviamo il risultato

N_FRAME_DA_PROCESSARE = 150  # ~5 secondi a 30fps
OUTPUT_VIDEO_DET = str(VIDEOS_DIR / 'day1_detection_pedoni.mp4')
OUTPUT_VIDEO_DET_TMP = str(VIDEOS_DIR / 'day1_detection_pedoni_tmp.mp4')
SOGLIA_CONF = 0.35

cap = cv2.VideoCapture(VIDEO_PEDONI)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO_DET_TMP, fourcc, fps, (w, h))

n_det_per_frame = []
for i in tqdm(range(N_FRAME_DA_PROCESSARE), desc='Detection'):
    ret, frame = cap.read()
    if not ret:
        break
    ris = model_yolo(frame, verbose=False)
    ann = disegna_detection(frame, ris, soglia_conf=SOGLIA_CONF, classi_filtro=[0])
    n = sum(1 for b in ris[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) > SOGLIA_CONF)
    n_det_per_frame.append(n)
    writer.write(ann)

cap.release()
writer.release()

OUTPUT_VIDEO_DET = converti_video_h264(OUTPUT_VIDEO_DET_TMP, OUTPUT_VIDEO_DET)
print(f'Video salvato in: {OUTPUT_VIDEO_DET}')
print(f'Media detection per frame: {np.mean(n_det_per_frame):.1f}')

In [ ]:
# Plot interattivo del numero di detection nel tempo
import plotly.graph_objects as go

media = np.mean(n_det_per_frame)
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=n_det_per_frame, mode='lines',
    line=dict(color='steelblue', width=1.5),
    name='Detection per frame'
))
fig.add_hline(
    y=media, line_dash='dash', line_color='red',
    annotation_text=f'Media: {media:.1f}', annotation_position='top right'
)
fig.update_layout(
    title='Detection nel tempo — MOT20-01',
    xaxis_title='Frame', yaxis_title='N° persone rilevate',
    height=350, margin=dict(l=50, r=20, t=50, b=40)
)
fig.show()

In [ ]:
# Mostra il video risultante
mostra_video_colab(OUTPUT_VIDEO_DET)

---
## 3. Multi-Object Tracking con BoxMOT

La **detection** risponde a: *"Dove sono gli oggetti in questo frame?"*  
Il **tracking** risponde a: *"Dove si trova lo stesso oggetto nel frame successivo?"*

**BoxMOT** è una libreria che combina detector (come YOLO) con algoritmi di tracking:
- **ByteTrack**: associa detection anche a bassa confidenza usando la predizione del moto
- **StrongSORT**: usa feature di re-identificazione visiva per tracking robusto
- **BoTrack**, **OCSORT**, e altri

Ogni oggetto tracciato riceve un **ID univoco** che persiste nel tempo.

**Output del tracker**: per ogni frame → `[x1, y1, x2, y2, track_id, conf, cls, ?]`

In [ ]:
from boxmot import ByteTrack
import torch

# Inizializzazione del tracker ByteTrack
# ByteTrack non richiede modelli aggiuntivi (solo motion-based)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device in uso: {device}')

tracker = ByteTrack(
    track_thresh=0.45,   # soglia minima confidenza per tracking
    match_thresh=0.8,    # soglia IoU per associazione
    track_buffer=30,     # frame da attendere prima di eliminare un track perso
)

In [ ]:
# Palette di colori per gli ID (per distinguere ogni oggetto)
np.random.seed(42)
PALETTE = np.random.randint(0, 255, size=(200, 3), dtype=np.uint8)

def colore_id(track_id: int) -> tuple:
    c = PALETTE[int(track_id) % len(PALETTE)]
    return int(c[0]), int(c[1]), int(c[2])

def disegna_tracks(frame: np.ndarray, tracks: np.ndarray) -> np.ndarray:
    """
    Disegna i bounding box con ID del tracker.
    tracks shape: (N, 7+) — [x1,y1,x2,y2, track_id, conf, cls, ...]
    """
    out = frame.copy()
    if tracks is None or len(tracks) == 0:
        return out
    for track in tracks:
        x1, y1, x2, y2 = map(int, track[:4])
        tid = int(track[4])
        conf = float(track[5])
        colore = colore_id(tid)
        cv2.rectangle(out, (x1, y1), (x2, y2), colore, 2)
        cv2.putText(out, f'ID:{tid}', (x1, y1 - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, colore, 2)
    return out

In [ ]:
# ── 3.1 Tracking completo su sequenza video ──────────────────────────────────
N_FRAME_TRACKING = 150
OUTPUT_VIDEO_TRACK = str(VIDEOS_DIR / 'day1_tracking_pedoni.mp4')
OUTPUT_VIDEO_TRACK_TMP = str(VIDEOS_DIR / 'day1_tracking_pedoni_tmp.mp4')
SOGLIA_YOLO = 0.3

tracker = ByteTrack()  # reset del tracker

cap = cv2.VideoCapture(VIDEO_PEDONI)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(OUTPUT_VIDEO_TRACK_TMP,
                         cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

id_attivi_per_frame = []

for i in tqdm(range(N_FRAME_TRACKING), desc='Tracking'):
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Detection con YOLO
    ris = model_yolo(frame, verbose=False, classes=[0])  # solo persone

    # 2. Conversione formato per BoxMOT: [x1,y1,x2,y2,conf,cls]
    dets = []
    for box in ris[0].boxes:
        conf = float(box.conf[0])
        if conf >= SOGLIA_YOLO:
            x1, y1, x2, y2 = map(float, box.xyxy[0])
            cls = int(box.cls[0])
            dets.append([x1, y1, x2, y2, conf, cls])

    dets_np = np.array(dets) if dets else np.empty((0, 6))

    # 3. Aggiornamento tracker
    tracks = tracker.update(dets_np, frame)
    id_attivi_per_frame.append(len(tracks) if tracks is not None else 0)

    # 4. Disegno e salvataggio
    frame_annotato = disegna_tracks(frame, tracks)

    # Overlay info
    cv2.putText(frame_annotato, f'Frame: {i+1} | Track attivi: {id_attivi_per_frame[-1]}',
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    writer.write(frame_annotato)

cap.release()
writer.release()

OUTPUT_VIDEO_TRACK = converti_video_h264(OUTPUT_VIDEO_TRACK_TMP, OUTPUT_VIDEO_TRACK)
print(f'Video salvato: {OUTPUT_VIDEO_TRACK}')
print(f'Media track attivi per frame: {np.mean(id_attivi_per_frame):.1f}')

In [ ]:
mostra_video_colab(OUTPUT_VIDEO_TRACK)

In [ ]:
# ── 3.2 Tracking su veicoli (UA-DETRAC) ─────────────────────────────────────
N_FRAME_VEI = 120
OUTPUT_VIDEO_TRACK_VEI = str(VIDEOS_DIR / 'day1_tracking_veicoli.mp4')
OUTPUT_VIDEO_TRACK_VEI_TMP = str(VIDEOS_DIR / 'day1_tracking_veicoli_tmp.mp4')

tracker_vei = ByteTrack()
cap = cv2.VideoCapture(VIDEO_VEICOLI)
fps_v = cap.get(cv2.CAP_PROP_FPS)
wv = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
hv = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(OUTPUT_VIDEO_TRACK_VEI_TMP,
                         cv2.VideoWriter_fourcc(*'mp4v'), fps_v, (wv, hv))

for i in tqdm(range(N_FRAME_VEI), desc='Tracking veicoli'):
    ret, frame = cap.read()
    if not ret:
        break
    ris = model_yolo(frame, verbose=False, classes=[2, 5, 7])  # car, bus, truck
    dets = []
    for box in ris[0].boxes:
        conf = float(box.conf[0])
        if conf >= 0.3:
            x1, y1, x2, y2 = map(float, box.xyxy[0])
            dets.append([x1, y1, x2, y2, conf, int(box.cls[0])])
    dets_np = np.array(dets) if dets else np.empty((0, 6))
    tracks = tracker_vei.update(dets_np, frame)
    frame_ann = disegna_tracks(frame, tracks)
    writer.write(frame_ann)

cap.release()
writer.release()

OUTPUT_VIDEO_TRACK_VEI = converti_video_h264(OUTPUT_VIDEO_TRACK_VEI_TMP, OUTPUT_VIDEO_TRACK_VEI)
mostra_video_colab(OUTPUT_VIDEO_TRACK_VEI)

### 🔍 Osservazioni sul tracking
- Notate come gli **ID rimangano stabili** anche quando un oggetto è parzialmente occluso?
- Cosa succede quando due persone si incrociano? (fenomeno di **ID switch**)
- In quali scene il tracker va in difficoltà?

---
## 4. Degradazione controllata con OpenCV

In scenari reali, le telecamere di sorveglianza spesso producono immagini di qualità ridotta per:
- Bassa risoluzione (telecamere economiche, compressione)
- Sfocatura (messa a fuoco errata, movimento rapido)
- Rumore (bassa illuminazione, sensori datati)

Simuliamo queste condizioni in modo **controllato e riproducibile** usando OpenCV.

**Obiettivo:** capire *quantitativamente* come ogni tipo di degradazione impatta la detection.

In [ ]:
# Prendiamo un frame di riferimento per gli esperimenti di degradazione
frame_ref = estrai_frame(VIDEO_PEDONI, n=1)[0]
print(f'Frame di riferimento: {frame_ref.shape[1]}x{frame_ref.shape[0]} px')

In [ ]:
# ── 4.1 Riduzione di risoluzione ─────────────────────────────────────────────
scale_factors = [1.0, 0.75, 0.5, 0.25]
frames_scala = [riduci_risoluzione(frame_ref, s) for s in scale_factors]
titoli_scala = [f'Scala {s:.0%}' for s in scale_factors]

mostra_griglia(frames_scala, titoli_scala, colonne=2)

In [ ]:
# ── 4.2 Blur gaussiano ───────────────────────────────────────────────────────
kernel_sizes = [0, 5, 15, 31]
frames_blur = [frame_ref.copy() if k == 0 else applica_blur(frame_ref, k)
               for k in kernel_sizes]
titoli_blur = [f'Kernel {k}x{k}' if k > 0 else 'Originale' for k in kernel_sizes]

mostra_griglia(frames_blur, titoli_blur, colonne=2)

In [ ]:
# ── 4.3 Rumore gaussiano ─────────────────────────────────────────────────────
livelli_rumore = [0, 15, 30, 60]
frames_rumore = [frame_ref.copy() if n == 0 else aggiungi_rumore(frame_ref, n)
                 for n in livelli_rumore]
titoli_rumore = [f'σ={n}' if n > 0 else 'Originale' for n in livelli_rumore]

mostra_griglia(frames_rumore, titoli_rumore, colonne=2)

In [ ]:
# ── 4.4 Degradazione combinata ───────────────────────────────────────────────
# Simuliamo una telecamera di sorveglianza reale a bassa qualità
frame_degradato_lieve  = degrada_immagine(frame_ref, scala=0.75, blur=5,  rumore=15.0)
frame_degradato_medio  = degrada_immagine(frame_ref, scala=0.5,  blur=11, rumore=25.0)
frame_degradato_forte  = degrada_immagine(frame_ref, scala=0.25, blur=21, rumore=40.0)

mostra_griglia(
    [frame_ref, frame_degradato_lieve, frame_degradato_medio, frame_degradato_forte],
    ['Originale', 'Degrado lieve', 'Degrado medio', 'Degrado forte'],
    colonne=2
)

---
## 5. Impatto della degradazione su Detection e Tracking

Ora applichiamo YOLO + ByteTrack alle stesse sequenze con degradazione crescente e **confrontiamo i risultati**.

In [ ]:
def run_detection_su_sequenza(video_path: str, n_frame: int,
                               degrado_fn=None, soglia: float = 0.35,
                               classi: list = [0]) -> dict:
    """
    Esegue YOLO su n_frame di un video, con eventuale degradazione.
    Restituisce statistiche aggregata.
    """
    cap = cv2.VideoCapture(video_path)
    n_det_lista = []
    conf_lista = []

    for _ in range(n_frame):
        ret, frame = cap.read()
        if not ret:
            break
        if degrado_fn is not None:
            frame = degrado_fn(frame)
        ris = model_yolo(frame, verbose=False, classes=classi)
        dets = [b for b in ris[0].boxes
                if int(b.cls[0]) in classi and float(b.conf[0]) >= soglia]
        n_det_lista.append(len(dets))
        conf_lista.extend([float(b.conf[0]) for b in dets])

    cap.release()
    return {
        'media_det': np.mean(n_det_lista) if n_det_lista else 0,
        'std_det': np.std(n_det_lista) if n_det_lista else 0,
        'media_conf': np.mean(conf_lista) if conf_lista else 0,
        'n_det_lista': n_det_lista,
    }

In [ ]:
# Definizione degli scenari di test
N_FRAME_TEST = 100

scenari = {
    'Originale':       None,
    'Scala 75%':       lambda f: riduci_risoluzione(f, 0.75),
    'Scala 50%':       lambda f: riduci_risoluzione(f, 0.50),
    'Scala 25%':       lambda f: riduci_risoluzione(f, 0.25),
    'Blur lieve':      lambda f: applica_blur(f, 7),
    'Blur forte':      lambda f: applica_blur(f, 21),
    'Rumore σ=25':     lambda f: aggiungi_rumore(f, 25),
    'Degrado medio':   lambda f: degrada_immagine(f, 0.5, 11, 25),
}

risultati_scenari = {}
for nome, fn in scenari.items():
    print(f'  Elaborazione: {nome}...')
    risultati_scenari[nome] = run_detection_su_sequenza(
        VIDEO_PEDONI, N_FRAME_TEST, degrado_fn=fn
    )

print('\n✅ Completato')

In [ ]:
# ── Tabella interattiva dei risultati ────────────────────────────────────────
import plotly.graph_objects as go
import pandas as pd

nomi        = list(risultati_scenari.keys())
medie       = [risultati_scenari[n]['media_det'] for n in nomi]
stds        = [risultati_scenari[n]['std_det']   for n in nomi]
confs       = [risultati_scenari[n]['media_conf'] for n in nomi]
baseline_det  = medie[0]
baseline_conf = confs[0]

# Colore cella in base al calo rispetto alla baseline
def _cella_color(val, baseline, invert=False):
    ratio = val / baseline if baseline > 0 else 1.0
    if invert:
        ratio = 1 - ratio + 1
    if ratio >= 0.9:
        return 'rgb(198,239,206)'   # verde
    elif ratio >= 0.7:
        return 'rgb(255,235,156)'   # giallo
    else:
        return 'rgb(255,199,206)'   # rosso

colors_det  = ['white'] + [_cella_color(m, baseline_det)  for m in medie[1:]]
colors_conf = ['white'] + [_cella_color(c, baseline_conf) for c in confs[1:]]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Scenario</b>', '<b>Media det/frame</b>', '<b>Dev. std</b>', '<b>Conf. media</b>'],
        fill_color='#2c3e50', font=dict(color='white', size=13),
        align='center', height=36
    ),
    cells=dict(
        values=[
            nomi,
            [f'{m:.1f}' for m in medie],
            [f'{s:.1f}' for s in stds],
            [f'{c:.3f}' for c in confs],
        ],
        fill_color=[
            ['white'] * len(nomi),
            colors_det,
            ['white'] * len(nomi),
            colors_conf,
        ],
        align='center', height=30,
        font=dict(size=12)
    )
)])
fig.update_layout(
    title='Impatto della degradazione su YOLO — riepilogo (🟢 ok · 🟡 calo lieve · 🔴 calo severo)',
    margin=dict(l=10, r=10, t=50, b=10), height=360
)
fig.show()

In [ ]:
# Grafici interattivi comparativi
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Numero di detection per scenario',
                                    'Confidenza media delle detection'))

# --- Grafico sinistra: media detection ---
fig.add_trace(
    go.Bar(
        y=nomi, x=medie,
        error_x=dict(type='data', array=stds),
        orientation='h',
        marker_color='steelblue', opacity=0.85,
        name='Media det/frame'
    ), row=1, col=1
)
fig.add_vline(x=baseline_det, line_dash='dash', line_color='red',
              annotation_text='Baseline', annotation_position='top right',
              row=1, col=1)

# --- Grafico destra: confidenza media ---
fig.add_trace(
    go.Bar(
        y=nomi, x=confs,
        orientation='h',
        marker_color='darkorange', opacity=0.85,
        name='Conf. media'
    ), row=1, col=2
)
fig.add_vline(x=baseline_conf, line_dash='dash', line_color='red',
              annotation_text='Baseline', annotation_position='top right',
              row=1, col=2)

fig.update_layout(
    height=420, showlegend=False,
    margin=dict(l=10, r=10, t=60, b=10)
)
fig.update_xaxes(title_text='Media det/frame', row=1, col=1)
fig.update_xaxes(title_text='Confidenza media', row=1, col=2)
fig.show()

In [ ]:
# ── Confronto visivo: detection originale vs. degradato ──────────────────────
frame_test = estrai_frame(VIDEO_PEDONI, n=1)[0]

confronti = [
    (frame_test, 'Originale'),
    (riduci_risoluzione(frame_test, 0.5), 'Scala 50%'),
    (applica_blur(frame_test, 21), 'Blur forte'),
    (degrada_immagine(frame_test, 0.5, 11, 25), 'Degrado medio'),
]

frames_vis = []
titoli_vis = []
for img, nome in confronti:
    ris = model_yolo(img, verbose=False, classes=[0])
    ann = disegna_detection(img, ris, soglia_conf=0.3, classi_filtro=[0])
    n = sum(1 for b in ris[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) > 0.3)
    frames_vis.append(ann)
    titoli_vis.append(f'{nome} ({n} det.)')

mostra_griglia(frames_vis, titoli_vis, colonne=2)

---
## 6. Riepilogo e discussione

### Cosa abbiamo fatto oggi:
1. **Esplorato** tre tipologie di dataset di sorveglianza (pedoni, veicoli, folla)
2. **Applicato** YOLOv8 per la detection di persone e veicoli
3. **Implementato** il Multi-Object Tracking con ByteTrack
4. **Simulato** degradazione controllata con OpenCV
5. **Quantificato** l'impatto della degradazione su detection e tracking

### Punti chiave:
- La **riduzione di risoluzione** è spesso più impattante del blur
- Il **tracker** può compensare parzialmente le mancate detection (usando la predizione del moto)
- La **soglia di confidenza** è un parametro critico: va calibrata in base al caso d'uso

### 🔍 Domande di riflessione:
- In quale scenario il tracking è più utile rispetto alla sola detection?
- Come gestireste un sistema in produzione dove la qualità del video varia?
- Quale tipo di degradazione è più difficile da gestire per YOLO?

In [ ]:
# ── ESERCIZIO FINALE: prova con un altro video ───────────────────────────────
# Scegliete un video dalla lista, cambiate i parametri e osservate i risultati

VIDEO_SCELTO = VIDEO_FOLLA  # ← modificate qui
SOGLIA_SCELTA = 0.35        # ← provate valori diversi
DEGRADO_SCELTO = None       # ← provate: lambda f: degrada_immagine(f, 0.5, 11, 20)

frame_es = estrai_frame(VIDEO_SCELTO, n=1)[0]
if DEGRADO_SCELTO:
    frame_es_deg = DEGRADO_SCELTO(frame_es)
else:
    frame_es_deg = frame_es

ris_es = model_yolo(frame_es_deg, verbose=False)
frame_es_ann = disegna_detection(frame_es_deg, ris_es, soglia_conf=SOGLIA_SCELTA)
n_es = sum(1 for b in ris_es[0].boxes if float(b.conf[0]) > SOGLIA_SCELTA)

print(f'Detection trovate: {n_es}')
mostra_confronto(frame_es, frame_es_ann, 'Originale', f'Detection (soglia={SOGLIA_SCELTA})')